In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
import plotly.graph_objects as go
import plotly.express as px

# Load raw data — Prophet works on raw sales, not engineered features
train = pd.read_csv('../data/raw/train.csv',
                    parse_dates=['Date'], low_memory=False)

# Filter to store 1 — open days only
STORE_ID = 1
store_df = train[
    (train['Store'] == STORE_ID) &
    (train['Open'] == 1) &
    (train['Sales'] > 0)
].copy()

# Prophet needs columns named ds (date) and y (target)
prophet_df = store_df[['Date', 'Sales']].rename(
    columns={'Date': 'ds', 'Sales': 'y'}
).sort_values('ds').reset_index(drop=True)

print(f'Store {STORE_ID}: {len(prophet_df)} days of sales data')
print(f'Date range: {prophet_df.ds.min()} to {prophet_df.ds.max()}')
print(f'Avg daily sales: EUR{prophet_df.y.mean():,.0f}')
prophet_df.head()

Store 1: 781 days of sales data
Date range: 2013-01-02 00:00:00 to 2015-07-31 00:00:00
Avg daily sales: EUR4,759


,ds,y
0,2013-01-02,5530
1,2013-01-03,4327
2,2013-01-04,4486
3,2013-01-05,4997
4,2013-01-07,7176


In [2]:
# Time-based split — last 6 weeks as test
# NEVER use random split on time series — it leaks future data
cutoff = prophet_df['ds'].max() - pd.Timedelta(weeks=6)
train_p = prophet_df[prophet_df['ds'] <= cutoff].copy()
test_p  = prophet_df[prophet_df['ds'] > cutoff].copy()

print(f'Train: {len(train_p)} days  |  Test: {len(test_p)} days')
print(f'Train ends: {train_p.ds.max().date()}')
print(f'Test starts: {test_p.ds.min().date()}')

# Train Prophet model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative'  # better for retail sales
)
model.fit(train_p)
print('Prophet training complete')

21:12:01 - cmdstanpy - INFO - Chain [1] start processing


Train: 745 days  |  Test: 36 days
Train ends: 2015-06-19
Test starts: 2015-06-20


21:12:02 - cmdstanpy - INFO - Chain [1] done processing


Prophet training complete


In [3]:
# Generate forecast — include historical + 42 days future
future = model.make_future_dataframe(periods=42)
forecast = model.predict(future)

# Evaluate on test set only
test_forecast = forecast[forecast['ds'].isin(test_p['ds'])]
merged = test_p.merge(
    test_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']],
    on='ds'
)

# MAPE — mean absolute percentage error
mape = np.mean(np.abs((merged['y'] - merged['yhat']) / merged['y'])) * 100
rmse = np.sqrt(np.mean((merged['y'] - merged['yhat'])**2))
mae  = np.mean(np.abs(merged['y'] - merged['yhat']))

print(f'Prophet results on Store {STORE_ID} — 6 week test:')
print(f'  MAPE : {mape:.2f}%')
print(f'  RMSE : EUR{rmse:,.0f}')
print(f'  MAE  : EUR{mae:,.0f}')

Prophet results on Store 1 — 6 week test:
  MAPE : 15.48%
  RMSE : EUR753
  MAE  : EUR667


In [8]:
# Full forecast plot with uncertainty bands
fig = go.Figure()

# Actual sales
fig.add_trace(go.Scatter(
    x=prophet_df['ds'], y=prophet_df['y'],
    name='Actual sales',
    line=dict(color='#378ADD', width=1)
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    name='Prophet forecast',
    line=dict(color='#1D9E75', width=2)
))

# Uncertainty band
fig.add_trace(go.Scatter(
    x=list(forecast['ds']) + list(forecast['ds'][::-1]),
    y=list(forecast['yhat_upper']) + list(forecast['yhat_lower'][::-1]),
    fill='toself',
    fillcolor='rgba(29,158,117,0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    name='Uncertainty band'
))

fig.update_layout(
    title=f'Store {STORE_ID} — Prophet forecast (MAPE: {mape:.1f}%)',
    xaxis_title='Date',
    yaxis_title='Sales (EUR)',
    height=450,
    legend=dict(orientation='h', y=-0.2)
)
fig.show()
import matplotlib.pyplot as plt
fig.write_html('../reports/prophet_forecast.html')
print('Saved as HTML -> reports/prophet_forecast.html')

Saved as HTML -> reports/prophet_forecast.html
